# Land Cover Classification using Sentinel-2 Satellite Images (Egypt)
### DEPI Data Science Project | Milestone 1 & Milestone 2

This notebook demonstrates the complete end-to-end pipeline for satellite image semantic segmentation using European Space Agency's **Sentinel-2** multi-spectral imagery. We implement:
1. **Data Exploration & Preprocessing (Milestone 1)**: Visualizing True Color, False Color, and NDVI maps, and analyzing class imbalances.
2. **Advanced Data Analysis (Milestone 2)**: Feature correlation using Mutual Information and dimensionality reduction using PCA.
3. **Model Selection & Training (Milestone 2)**: Designing and training a U-Net convolutional neural network to classify pixels into 5 classes: *Trees/Forest, Agriculture, Desert, Water, and Urban/Roads*.

## 1. Setup and Environment Verification
First, we import the necessary packages and verify if PyTorch is configured with CUDA/GPU support for training.

In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# Add the parent directory to the path so we can import our modules
sys.path.append(os.path.abspath('..'))

from src.dataset import get_stratified_splits, Sentinel2SegmentationDataset
from src.features import calculate_ndvi, apply_pca_to_image, analyze_band_importance
from src.model import UNet
from src.train import run_training_pipeline
from src.utils import plot_spectral_signatures, plot_training_history, visualize_predictions

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Milestone 1: Data Exploration, Preprocessing & Feature Engineering
We start by performing a stratified split on our dataset. The dataset contains 719 `.tif` files representing different locations in Egypt (Alexandria, Cairo, Fayoum, Luxor, Mansoura, Aswan, and Siwa). To ensure that our training, validation, and test splits are representative, we stratify the split using the geographical region prefix of the filenames.

In [ ]:
dataset_dir = "../egypt_s2_diverse_dataset"

# Perform stratified region-based splitting
train_paths, val_paths, test_paths = get_stratified_splits(dataset_dir)

### 2.1 Visualizing Satellite Composites & NDVI
Sentinel-2 images contain 12 spectral bands. Let's load a sample image from the Western Desert (Siwa Oasis) and Fayoum, and plot:
1. **True Color RGB** (Bands 4, 3, 2) normalized to $[0, 1]$
2. **False Color Infrared** (Bands 8, 4, 3) which makes vegetation appear bright red
3. **NDVI Map** showing vegetation density
4. **Ground Truth Label Map** showing the 5 land cover classes

In [ ]:
sample_file = os.path.join(dataset_dir, "HawaraFayoum_000_30.932_29.252.tif")

with rasterio.open(sample_file) as src:
    blue = src.read(2)
    green = src.read(3)
    red = src.read(4)
    nir = src.read(8)
    label = src.read(13)

# Normalization helper
def norm(band):
    vmin, vmax = np.percentile(band, [2, 98])
    if vmax - vmin > 0:
        return np.clip((band - vmin) / (vmax - vmin), 0.0, 1.0)
    return np.zeros_like(band)

rgb = np.stack([norm(red), norm(green), norm(blue)], axis=-1)
false_color = np.stack([norm(nir), norm(red), norm(green)], axis=-1)
ndvi = calculate_ndvi(red, nir)

# Plotting
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0, 0].imshow(rgb)
axes[0, 0].set_title("True Color RGB")
axes[0, 0].axis('off')

axes[0, 1].imshow(false_color)
axes[0, 1].set_title("False Color Composite (Vegetation appears Red)")
axes[0, 1].axis('off')

im_ndvi = axes[1, 0].imshow(ndvi, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1, 0].set_title("NDVI Map")
axes[1, 0].axis('off')
fig.colorbar(im_ndvi, ax=axes[1, 0], orientation='horizontal', pad=0.05)

cmap = plt.get_cmap('tab10')
im_label = axes[1, 1].imshow(label, cmap=cmap, vmin=-0.5, vmax=4.5)
axes[1, 1].set_title("Label Map (Ground Truth)")
axes[1, 1].axis('off')
cbar = fig.colorbar(im_label, ax=axes[1, 1], orientation='horizontal', pad=0.05, ticks=[0, 1, 2, 3, 4])
cbar.ax.set_xticklabels(['Trees/Forest (0)', 'Agriculture (1)', 'Desert (2)', 'Water (3)', 'Urban/Roads (4)'])

plt.suptitle("Sample Exploratory Data Visualization (Fayoum Oasis)", fontsize=16)
plt.tight_layout()
plt.show()

### 2.2 Spectral Signatures of Land Cover Types
Let's visualize the average reflectance profile (or spectral signature) across all 12 bands for each class. This represents how each material (sand, water, vegetation, buildings) reflects light across different parts of the spectrum.

In [ ]:
class_signatures = {
    0: [989.92, 1154.39, 1557.54, 1872.01, 2262.78, 2638.05, 2822.39, 2905.30, 2943.95, 2952.63, 2920.77, 2445.71],  # Trees/Forest
    1: [604.65, 696.70, 1087.43, 1136.53, 1712.72, 2841.65, 3239.96, 3359.20, 3431.43, 3452.87, 2460.33, 1676.72],    # Agriculture
    2: [1529.18, 1912.10, 2669.42, 3476.46, 3836.05, 3929.06, 4041.53, 4114.59, 4103.97, 4146.97, 4820.83, 4363.43],  # Desert
    3: [568.54, 553.66, 828.49, 649.37, 801.28, 788.07, 851.88, 809.23, 853.06, 1219.69, 842.16, 675.76],             # Water
    4: [1172.15, 1418.38, 1870.29, 2280.90, 2628.20, 2968.75, 3138.94, 3224.17, 3260.68, 3303.83, 3419.48, 3018.46]   # Urban/Roads
}

band_names = ["B01", "B02", "B03", "B04", "B05", "B06", "B07", "B08", "B8A", "B09", "B11", "B12"]
class_names = {
    0: "Trees/Forest", 1: "Agriculture/Vegetation", 2: "Desert", 3: "Water", 4: "Urban/Roads"
}
colors = {
    0: "forestgreen", 1: "limegreen", 2: "orange", 3: "dodgerblue", 4: "dimgray"
}

plt.figure(figsize=(12, 6))
for c, sig in class_signatures.items():
    plt.plot(band_names, sig, label=class_names[c], color=colors[c], marker='o', linewidth=2.5, markersize=8)
plt.title("Sentinel-2 Spectral Signatures in Egypt", fontsize=14, fontweight='bold')
plt.xlabel("Spectral Bands", fontsize=12)
plt.ylabel("Mean Reflectance", fontsize=12)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 3. Milestone 2: Advanced Data Analysis
### 3.1 Mutual Information Band Importance
We calculate the Mutual Information (MI) score between each of the 12 spectral bands and the pixel land cover labels. Higher MI indicates that the band carries more information about the class categories.

In [ ]:
# Sample every 50th file to make Mutual Info fast in notebook
sampled_mi_files = [os.path.basename(fp) for fp in train_paths[::20]]
mi_scores = analyze_band_importance(dataset_dir, sampled_mi_files, n_pixels_per_file=1000)

bands, scores = zip(*mi_scores)
plt.figure(figsize=(10, 5))
plt.bar(bands, scores, color='darkslateblue', edgecolor='black', alpha=0.8)
plt.title("Sentinel-2 Band Importance for Land Cover Classification", fontsize=12, fontweight='bold')
plt.xlabel("Spectral Band", fontsize=11)
plt.ylabel("Mutual Information Score", fontsize=11)
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

### 3.2 Dimensionality Reduction (PCA)
We apply Principal Component Analysis (PCA) across the 12 bands to compress the spectral features into 3 principal components explaining the majority of the variance, and project them as an RGB composite image.

In [ ]:
# Load image data
with rasterio.open(sample_file) as src:
    image_data = src.read(list(range(1, 13)))

pca_image, variance_ratios = apply_pca_to_image(image_data, n_components=3)
print("Explained Variance per Principal Component:")
for comp, var in enumerate(variance_ratios, 1):
    print(f"  PC {comp}: {var*100:.2f}%")

# Normalize components for display
def norm_comp(comp):
    c_min, c_max = comp.min(), comp.max()
    return (comp - c_min) / (c_max - c_min) if c_max - c_min > 0 else np.zeros_like(comp)

pca_rgb = np.stack([norm_comp(pca_image[0]), norm_comp(pca_image[1]), norm_comp(pca_image[2])], axis=-1)

plt.figure(figsize=(8, 8))
plt.imshow(pca_rgb)
plt.title(f"PCA Color Composite (PC1, PC2, PC3)\nTotal Variance Explained: {sum(variance_ratios)*100:.2f}%")
plt.axis('off')
plt.show()

## 4. Milestone 2: Model Selection & Training (U-Net)
We train the model using our custom U-Net segmentation architecture. We run the training pipeline for 10 epochs. The training pipeline uses an inverse-frequency class-weighted cross-entropy loss function to address class imbalance, standard AdamW optimizer, and Cosine Annealing learning rate scheduler.

In [ ]:
checkpoint_dir = "../models"

# Run training (using 10 epochs as a solid standard)
history, test_class_ious = run_training_pipeline(
    dataset_dir=dataset_dir,
    checkpoint_dir=checkpoint_dir,
    epochs=10,
    batch_size=8,
    lr=1e-4,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
)

### 4.1 Plotting Training History
Let's plot the training/validation loss, pixel accuracy, and mean IoU history.

In [ ]:
epochs = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, history['train_loss'], label='Train Loss', color='royalblue', marker='o')
axes[0].plot(epochs, history['val_loss'], label='Val Loss', color='crimson', marker='s')
axes[0].set_title('Loss History')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

axes[1].plot(epochs, [x*100 for x in history['train_acc']], label='Train Acc', color='royalblue', marker='o')
axes[1].plot(epochs, [x*100 for x in history['val_acc']], label='Val Acc', color='crimson', marker='s')
axes[1].set_title('Pixel Accuracy History (%)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

axes[2].plot(epochs, [x*100 for x in history['val_miou']], label='Val mIoU', color='forestgreen', marker='^')
axes[2].set_title('Validation Mean IoU History (%)')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('mIoU (%)')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend()

plt.tight_layout()
plt.show()

### 4.2 Model Inference & Visual Evaluations
We load our best model checkpoint and run inference on 3 random test dataset samples to inspect the model's segmentation outputs visually compared to the ground truth mask.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(n_channels=12, n_classes=5).to(device)

best_checkpoint = os.path.join(checkpoint_dir, "best_unet_model.pth")
if os.path.exists(best_checkpoint):
    checkpoint = torch.load(best_checkpoint, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']} with Val mIoU {checkpoint['val_miou']*100:.2f}%")

test_dataset = Sentinel2SegmentationDataset(test_paths, augment=False)

# Perform visualizations directly in the notebook
indices = np.random.choice(len(test_dataset), 3, replace=False)
classes = ["Trees/Forest (0)", "Agriculture/Veg (1)", "Desert (2)", "Water (3)", "Urban/Roads (4)"]
cmap = plt.get_cmap('tab10')

model.eval()
with torch.no_grad():
    for i, idx in enumerate(indices):
        img, mask = test_dataset[idx]
        inputs = img.unsqueeze(0).to(device)
        outputs = model(inputs)
        pred = torch.argmax(outputs, dim=1).squeeze(0).cpu().numpy()
        
        img_np = img.numpy()
        mask_np = mask.numpy()
        
        red = img_np[3]
        green = img_np[2]
        blue = img_np[1]
        
        rgb = np.stack([norm(red), norm(green), norm(blue)], axis=-1)
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(rgb)
        axes[0].set_title("True Color RGB")
        axes[0].axis('off')
        
        axes[1].imshow(mask_np, cmap=cmap, vmin=-0.5, vmax=4.5)
        axes[1].set_title("Ground Truth Label")
        axes[1].axis('off')
        
        im_pred = axes[2].imshow(pred, cmap=cmap, vmin=-0.5, vmax=4.5)
        axes[2].set_title("Model Prediction")
        axes[2].axis('off')
        
        cbar = fig.colorbar(im_pred, ax=axes.ravel().tolist(), orientation='horizontal', pad=0.08, ticks=[0, 1, 2, 3, 4])
        cbar.ax.set_xticklabels(classes)
        
        plt.suptitle(f"Sample Prediction Test (Dataset Index: {idx})", fontsize=14)
        plt.show()